<a href="https://colab.research.google.com/github/somendrew/RAG_System/blob/main/Conversational_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 📘 Conversational RAG: Key Concepts

### Problem with Basic RAG:

*   **Statelessness**: Each query is independent. Follow-up questions like "What causes that?" lack context, leading to irrelevant retrieval because "that" has no prior meaning for the retriever.

### Conversational RAG Solutions:

1.  **History-Aware Retrieval (Query Rewriting)**:
    *   **Goal**: Transform ambiguous follow-up questions into standalone queries.
    *   **Process**: LLM rewrites the question using chat history (e.g., "What causes that?" becomes "What causes the Moon to show different phases?").
    *   **Benefit**: Ensures more accurate retrieval by providing clear context.

2.  **Chat History in Answer Prompt**:
    *   **Goal**: Maintain conversational flow and improve answer relevance.
    *   **Process**: The LLM generates the final answer with the full chat history as context.
    *   **Benefit**: Helps maintain conversational tone/context and improves retrieval accuracy by allowing the LLM to understand the broader dialogue.

### Key LangChain Components:

*   **`create_history_aware_retriever`**: Wraps your retriever to add the "contextualize the question" step using an LLM.
*   **`create_stuff_documents_chain`**: Generates the final answer using retrieved documents, the question, and chat history.
*   **`create_retrieval_chain`**: Connects the history-aware retriever and the stuff-documents chain into a complete pipeline.

### Memory Management:

*   **Session-Based Memory**: Uses an in-memory dictionary (keyed by `session_id`) to track separate conversations.
*   **Purpose**: Allows multiple users or independent interactions to maintain distinct chat histories.
*   **Note**: This is a "chain-based memory pattern." While LangGraph offers a more modern approach, understanding this method is fundamental.

## The Conversational RAG Approach: Step-by-Step

Here's a detailed breakdown of how Conversational RAG operates:

### Step-by-Step Process:

1.  **Store the Conversation History:**
    *   Maintain a running list of all turns in the conversation (e.g., `[Human: "Why does the Moon...", AI: "The Moon shows phases because...", Human: "What causes that exactly?"]`).
    *   This is a simple chronological record of interaction, growing with each new turn.

2.  **Rewrite the Question using History:**
    *   **Before retrieval**, an additional LLM call is made.
    *   Its sole purpose is to **rewrite the new question into a standalone query**, leveraging the entire chat history.
    *   Example: "What causes that exactly?" becomes "What causes the Moon's phases?".
    *   This rewritten version (not the original) is then used for embedding and searching.

3.  **Retrieve Information:**
    *   Perform retrieval as normal, but crucially, use the **rewritten question** for the search.

4.  **Generate the Answer:**
    *   Feed both the **retrieved document chunks** and the **full conversation history** into the LLM.
    *   This allows the LLM to generate an answer that is not only factually supported but also maintains the correct tone and conversational context (e.g., recognizing it's mid-conversation about the Moon).

5.  **Append Current Turn to History:**
    *   After generating the response, append the current question-answer pair to the conversation history.
    *   This ensures the history is continuously updated and available for subsequent questions.

### Why an Extra LLM Call for Rewriting is Necessary:

*   **Avoids Noise**: Simply retrieving with the entire conversation history can be noisy, as early, irrelevant turns might pollute the search.
*   **Handles Ambiguity**: It effectively resolves pronouns and contextual references (like "that") that would otherwise confuse the retriever.
*   **Efficiency**: A dedicated rewrite step, typically a small LLM call, is very cheap in terms of computational resources.
*   **Precision**: It dramatically improves retrieval precision by providing a clear, unambiguous query.

### The "Plumbing" - Bookkeeping for Conversational RAG:

These are not new concepts but essential logistical components:

*   **History Storage**: A mechanism to store conversation history per user or session (e.g., a dictionary mapping `session_id` to a list of messages).
*   **Auto-Appending**: A system to automatically add new conversation turns to the history after each interaction.
*   **LangChain's `RunnableWithMessageHistory`**: This is a powerful wrapper that automates steps 1 and 5 (storing and appending history), significantly reducing manual bookkeeping.

### Core Innovations:

Ultimately, there are only two genuinely novel ideas introduced by Conversational RAG:

1.  **Rewriting the question** using history **before** retrieval.
2.  Maintaining a **growing list of conversation turns**, uniquely identified by session.

# Implimentation

# 1. Installation

In [14]:
!pip install -q langchain langchain-openai openai langchain-text-splitters langchain-core langchain-community langchain-classic langchain-chroma chromadb


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 113.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the 

# 2. API SETUP

In [10]:
from google.colab import userdata

# Retrieve the OpenAI API key from Colab's secrets manager.
# The key should be stored under the name 'OPENAI_API_KEY'.
OPENAI_API_KEY = userdata.get('api_key')

# 3. Load Corpus

In [11]:
import requests
from langchain_core.documents import Document

def fetch_wikipedia_page(title: str) -> Document:
    """
    Fetches a Wikipedia page's extract as a LangChain Document.

    Args:
        title (str): The title of the Wikipedia page to fetch.

    Returns:
        Document: A LangChain Document object containing the page content and metadata.
    """
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",       # Action to perform: query the API.
        "prop": "extracts",      # Request page extracts.
        "explaintext": True,     # Return plain text instead of HTML.
        "titles": title,         # The title of the page to query.
        "format": "json",        # Response format: JSON.
        "redirects": 1,          # Resolve redirects.
    }
    headers = {"User-Agent": "RAG-Learning-Notebook/1.0 (educational use)"}

    resp = requests.get(url, params=params, headers=headers, timeout=10)
    resp.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx).
    data = resp.json()

    pages = data["query"]["pages"]
    page = next(iter(pages.values()))
    text = page.get("extract", "")

    return Document(page_content=text, metadata={"title": page.get("title", title), "source": url})

topics = ["Solar System","Moon"]

docs = [fetch_wikipedia_page(t) for t in topics]

for d in docs:
    print(d.metadata, "-", len(d.page_content), "chars")

{'title': 'Solar System', 'source': 'https://en.wikipedia.org/w/api.php'} - 61733 chars
{'title': 'Moon', 'source': 'https://en.wikipedia.org/w/api.php'} - 89900 chars


# 4. Split into chunks

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks")

Split into 251 chunks


# 5. Embed + store in Chroma → retriever


In [16]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=OPENAI_API_KEY)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="conversational_rag_demo",
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# 6. LLM

In [17]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model = "gpt-4o-mini",
    temperature = 0,
    api_key = OPENAI_API_KEY,
    max_tokens = 300,
    use_responses_api= False,
    )

# Conversational RAG Layer

### Step 1: Session Based chat history

In [18]:
from langchain_community.chat_message_histories import ChatMessageHistory

store = {} # session_id -> ChatMessageHistory

def get_session_history(session_id):
  if session_id not in store:
    store[session_id] = ChatMessageHistory()
  return store[session_id]




### Step 2: History-aware retriever — rewrites question using history

In [28]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

# --- Step 2: question rewriting, as an explicit LCEL chain ---
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the user's latest question as a standalone question, "
               "using the chat history for context. Do not answer it. "
               "If it's already standalone, return it unchanged."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

query_rewriter = rewrite_prompt | llm | StrOutputParser()


### Step 3 & 4: Retrieve, then generate the answer (using history + retrieved chunks)

In [29]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# --- Steps 3 & 4: retrieve using rewritten query, then answer ---
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY this context. Say so if you don't know.\n\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

def retrieve_and_answer(inputs):
    # inputs = {"input": ..., "chat_history": [...]}
    standalone_question = query_rewriter.invoke(inputs)
    retrieved_docs = retriever.invoke(standalone_question)
    context = format_docs(retrieved_docs)

    answer = (answer_prompt | llm | StrOutputParser()).invoke({
        "input": inputs["input"],
        "chat_history": inputs["chat_history"],
        "context": context,
    })

    return {"answer": answer, "context": retrieved_docs, "input": inputs["input"]}

rag_chain = RunnableLambda(retrieve_and_answer)

## Step 5: Conversational RAG

In [31]:
from langchain_core.runnables.history import RunnableWithMessageHistory

conversational_rag = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


### Step 6: Run a conversation

In [33]:
session_id = "user_1"

r1 = conversational_rag.invoke(
    {"input": "Why does the Moon show different phases?"},
    config={"configurable": {"session_id": session_id}},
)

r2 = conversational_rag.invoke(
    {"input": "What causes that exactly?"},
    config={"configurable": {"session_id": session_id}},
)

r3 = conversational_rag.invoke(
    {"input": "Does Earth have a similar phenomenon?"},
    config={"configurable": {"session_id": session_id}},
)

def show(r, label):
    print(f"{label} Q: {r['input']}")
    print(f"{label} A: {r['answer']}")
    print()

show(r1, "1")
show(r2, "2")
show(r3, "3")

1 Q: Why does the Moon show different phases?
1 A: The Moon shows different phases due to its rotation as it orbits Earth, which changes its orientation toward the Sun. This results in varying illumination of the Moon by the Sun, observable from Earth as the changing lunar phases.

2 Q: What causes that exactly?
2 A: The different phases of the Moon are caused by the relative positions of the Moon, Earth, and Sun. As the Moon orbits Earth, the angle at which sunlight illuminates the Moon changes, leading to the various phases we observe from Earth. This cycle of phases is known as a lunation.

3 Q: Does Earth have a similar phenomenon?
3 A: The context does not provide information about whether Earth has a similar phenomenon to the Moon's phases. Therefore, I don't know.



### Step 7: Inspect the memory

In [34]:
for msg in store[session_id].messages:
    print(f"{msg.type}: {msg.content[:100]}")

human: Why does the Moon show different phases?
ai: The Moon shows different phases due to its rotation as it orbits Earth, which changes its orientatio
human: Why does the Moon show different phases?
ai: The Moon shows different phases because it rotates as it orbits Earth, changing its orientation towa
human: What causes that exactly?
ai: The different phases of the Moon are caused by the relative positions of the Moon, Earth, and Sun. A
human: Does Earth have a similar phenomenon?
ai: The context does not provide information about whether Earth has a similar phenomenon to the Moon's 
human: Why does the Moon show different phases?
ai: The Moon shows different phases due to its rotation as it orbits Earth, which changes its orientatio
human: What causes that exactly?
ai: The different phases of the Moon are caused by the relative positions of the Moon, Earth, and Sun. A
human: Does Earth have a similar phenomenon?
ai: The context does not provide information about whether Earth has a